In [ ]:
```xml
<VSCode.Cell language="markdown">
# 🔄 Data Pipeline & ETL Architecture Guide

**Building robust, scalable data processing workflows**

This guide covers:
- ETL vs ELT patterns
- Apache Spark and Airflow
- Data validation and quality
- Incremental loading strategies
- Error handling and recovery
- Monitoring data pipelines
- Cost optimization
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 1. ETL vs ELT

### ETL (Extract, Transform, Load)
```
Source → Extract → Transform → Validate → Load → Target

Traditional approach:
- Transform in middleware
- Clean data before loading
- Suitable for smaller volumes
- Example: Talend, Informatica
```

### ELT (Extract, Load, Transform)
```
Source → Extract → Load → Transform → Target

Modern cloud approach:
- Load raw data first
- Transform in data warehouse
- Scale transformations easily
- Example: Snowflake, BigQuery, Spark on Databricks
```

### Decision Matrix
```
Use ETL when:
- Small data volumes (<1TB)
- Complex transformations
- Multiple targets
- Compliance requirements

Use ELT when:
- Large data volumes (>1TB)
- Cloud data warehouse available
- Fast ingestion needed
- Flexible schema required
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 2. Apache Spark Pipelines

```python
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, sum as spark_sum
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType

# Initialize Spark
spark = SparkSession.builder \
    .appName("ETLPipeline") \
    .config("spark.sql.shuffle.partitions", 200) \
    .getOrCreate()

# 1. EXTRACT - Read from source
def extract_data():
    """Extract from multiple sources"""
    
    # From Parquet
    users_df = spark.read.parquet("s3://bucket/users/")
    
    # From CSV
    orders_df = spark.read.csv(
        "s3://bucket/orders/",
        header=True,
        inferSchema=True
    )
    
    # From database
    transactions_df = spark.read \
        .format("jdbc") \
        .option("url", "jdbc:postgresql://db:5432/app") \
        .option("dbtable", "transactions") \
        .option("user", "spark") \
        .option("password", "password") \
        .load()
    
    return users_df, orders_df, transactions_df

# 2. TRANSFORM
def transform_data(users_df, orders_df, transactions_df):
    """Apply transformations"""
    
    # Join data
    enriched_df = orders_df.join(
        users_df,
        orders_df.user_id == users_df.id,
        "left"
    )
    
    # Data cleaning
    enriched_df = enriched_df \
        .filter(col("order_total") > 0) \
        .fillna({"discount": 0}) \
        .dropDuplicates(["order_id"])
    
    # Add calculations
    enriched_df = enriched_df.withColumn(
        "net_amount",
        col("order_total") - col("discount")
    )
    
    # Aggregate
    daily_summary = enriched_df \
        .groupBy("user_id", "order_date") \
        .agg(
            spark_sum("net_amount").alias("daily_total"),
            spark_sum("order_total").alias("revenue")
        )
    
    return enriched_df, daily_summary

# 3. VALIDATE
def validate_data(df):
    """Data quality checks"""
    
    # Check for nulls
    null_count = df.filter(col("user_id").isNull()).count()
    if null_count > 0:
        raise Exception(f"Found {null_count} null user_ids")
    
    # Check ranges
    invalid_amounts = df.filter(col("order_total") < 0).count()
    if invalid_amounts > 0:
        raise Exception(f"Found {invalid_amounts} negative amounts")
    
    # Check uniqueness
    duplicate_orders = df.groupBy("order_id").count().filter(col("count") > 1)
    if duplicate_orders.count() > 0:
        raise Exception("Found duplicate orders")
    
    print("✓ Data validation passed")

# 4. LOAD
def load_data(df, target):
    """Load to target"""
    
    if target == "parquet":
        df.write.mode("overwrite").parquet("s3://bucket/output/orders/")
    
    elif target == "warehouse":
        df.write \
            .format("bigquery") \
            .option("table", "project.dataset.orders") \
            .mode("overwrite") \
            .save()
    
    elif target == "database":
        df.write \
            .format("jdbc") \
            .option("url", "jdbc:postgresql://db:5432/app") \
            .option("dbtable", "orders_processed") \
            .option("user", "spark") \
            .mode("overwrite") \
            .save()

# Execute pipeline
if __name__ == "__main__":
    users, orders, transactions = extract_data()
    enriched, summary = transform_data(users, orders, transactions)
    
    validate_data(enriched)
    validate_data(summary)
    
    load_data(enriched, "warehouse")
    load_data(summary, "parquet")
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 3. Apache Airflow DAGs

```python
from airflow import DAG
from airflow.operators.bash import BashOperator
from airflow.operators.python import PythonOperator
from airflow.providers.amazon.aws.operators.s3 import S3CopyObjectOperator
from airflow.utils.dates import days_ago
from datetime import timedelta

# Define DAG
default_args = {
    'owner': 'data_team',
    'retries': 2,
    'retry_delay': timedelta(minutes=5),
    'start_date': days_ago(1),
    'email_on_failure': ['alerts@company.com']
}

dag = DAG(
    'daily_etl_pipeline',
    default_args=default_args,
    description='Daily ETL processing',
    schedule_interval='0 2 * * *',  # 2 AM daily
    catchup=False
)

# Tasks
def extract_task(**context):
    """Extract data"""
    import boto3
    s3 = boto3.client('s3')
    s3.download_file('bucket', 'raw/orders.csv', '/tmp/orders.csv')
    context['task_instance'].xcom_push(key='source_file', value='/tmp/orders.csv')

def validate_task(**context):
    """Validate extracted data"""
    import pandas as pd
    
    source_file = context['task_instance'].xcom_pull(
        task_ids='extract',
        key='source_file'
    )
    
    df = pd.read_csv(source_file)
    
    # Validation checks
    assert len(df) > 0, "Empty dataset"
    assert df['order_id'].nunique() == len(df), "Duplicate orders"
    assert (df['amount'] >= 0).all(), "Negative amounts"
    
    print("✓ Validation passed")

# Operators
extract = PythonOperator(
    task_id='extract',
    python_callable=extract_task,
    dag=dag
)

spark_transform = BashOperator(
    task_id='transform',
    bash_command='spark-submit /opt/spark/jobs/etl.py',
    dag=dag
)

validate = PythonOperator(
    task_id='validate',
    python_callable=validate_task,
    dag=dag
)

load = BashOperator(
    task_id='load',
    bash_command='python /opt/jobs/load_to_warehouse.py',
    dag=dag
)

notify = BashOperator(
    task_id='notify',
    bash_command='curl -X POST http://alerts/pipeline-complete',
    dag=dag,
    trigger_rule='all_done'
)

# Dependencies
extract >> spark_transform >> validate >> load >> notify
```

### Error Handling in Airflow
```python
from airflow.models import Variable
from airflow.exceptions import AirflowFailException

def handle_errors(**context):
    """Handle pipeline errors"""
    execution_date = context['execution_date']
    task_instance = context['task_instance']
    
    # Log error
    print(f"Task {task_instance.task_id} failed on {execution_date}")
    
    # Notify
    import smtplib
    message = f"ETL failed: {task_instance.log_url}"
    # Send email...
    
    # Retry logic
    if task_instance.try_number < 3:
        raise AirflowFailException("Retrying...")
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 4. Incremental Loading & Deduplication

```python
from datetime import datetime, timedelta

def incremental_load(last_checkpoint: datetime):
    """Load only new data since last run"""
    
    # Query incremental data
    query = f"""
    SELECT * FROM source_table
    WHERE updated_at > '{last_checkpoint}'
    ORDER BY updated_at
    """
    
    source_data = spark.sql(query)
    
    # Read existing target
    target_data = spark.read.parquet("s3://bucket/target/")
    
    # Combine and deduplicate
    combined = source_data.union(target_data)
    
    deduplicated = combined \
        .dropDuplicates(["id"]) \
        .sort("updated_at", ascending=False) \
        .drop_duplicates(subset=["id"])
    
    # Write back
    deduplicated.write.mode("overwrite").parquet("s3://bucket/target/")
    
    # Save checkpoint
    Variable.set("last_etl_checkpoint", datetime.now().isoformat())
```
</MySCode.Cell>

<MySCode.Cell language="markdown">
## 5. Data Pipeline Monitoring & Checklist

✅ **Pipeline Design**
- [ ] Clear source/target definitions
- [ ] Incremental loading strategy
- [ ] Error handling and retry logic
- [ ] Data validation checks

✅ **Performance**
- [ ] Partition strategies
- [ ] Compression settings
- [ ] Resource allocation
- [ ] Cost tracking

✅ **Reliability**
- [ ] Idempotent operations
- [ ] Dead letter queues
- [ ] Checkpointing
- [ ] Recovery procedures

✅ **Monitoring**
- [ ] Data quality metrics
- [ ] Pipeline duration tracking
- [ ] Alert thresholds
- [ ] Audit logging
</MySCode.Cell>
```